# 02 — Model Training

This notebook walks through training a YOLOv8 model on the custom dataset, including parameter configuration, training execution, and loss curve visualization.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.visualization import plot_training_curves

## Configuration

In [ ]:
# Training configuration
DATA_YAML = '../configs/data.yaml'
MODEL_NAME = 'yolov8n.pt'       # YOLOv8 nano — fast, good for experimentation
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16                  # Adjust based on GPU memory
OPTIMIZER = 'AdamW'
PROJECT = '../runs/train'
EXPERIMENT_NAME = 'yolov8n_custom'

print(f'Model:     {MODEL_NAME}')
print(f'Epochs:    {EPOCHS}')
print(f'Image Size: {IMG_SIZE}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Optimizer: {OPTIMIZER}')

## Step 1 — Verify Pretrained Baseline

Before fine-tuning, make sure the pretrained model loads and runs inference correctly.

In [ ]:
# Load pretrained model
model = YOLO(MODEL_NAME)
print(f'Model loaded: {MODEL_NAME}')
print(f'Model type:  {model.type}')

# Quick sanity check — predict on a sample image if available
sample_dir = Path('../data/dataset/train/images')
samples = list(sample_dir.glob('*.[jp][pn]g'))[:1] if sample_dir.exists() else []
if samples:
    results = model.predict(str(samples[0]), verbose=False)
    print(f'Sanity check: {len(results[0].boxes)} detections on sample image')
else:
    print('No sample images found for sanity check')

## Step 2 — Train the Model

In [ ]:
# Train the model
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    optimizer=OPTIMIZER,
    project=PROJECT,
    name=EXPERIMENT_NAME,
    patience=10,          # Early stopping patience
    save=True,
    plots=True,           # Generate built-in plots
    verbose=True,
)

print('\nTraining complete!')
print(f'Best model saved at: {PROJECT}/{EXPERIMENT_NAME}/weights/best.pt')

## Step 3 — Training Curves

In [ ]:
# Plot training curves from results.csv
results_csv = Path(f'{PROJECT}/{EXPERIMENT_NAME}/results.csv')

if results_csv.exists():
    plot_training_curves(results_csv, Path('../outputs/plots'))
    
    # Also display key metrics
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print('\nFinal epoch metrics:')
    print(df.tail(1).to_string(index=False))
else:
    print('No results.csv found — run training first.')

## Step 4 — Validate the Trained Model

In [ ]:
# Validate on validation set
best_model_path = Path(f'{PROJECT}/{EXPERIMENT_NAME}/weights/best.pt')

if best_model_path.exists():
    trained_model = YOLO(str(best_model_path))
    val_results = trained_model.val(data=DATA_YAML)
    
    print(f"\nValidation Results:")
    print(f"  mAP@50:    {val_results.box.map50:.4f}")
    print(f"  mAP@50-95: {val_results.box.map:.4f}")
else:
    print('No trained model found — run training first.')

## Notes

- **YOLOv8n** is recommended for initial experiments (fast training, low GPU memory)
- **YOLOv8s** can be tried for better accuracy if hardware allows
- Monitor the loss curves to detect overfitting (validation loss increasing while training loss decreases)
- Copy `best.pt` to `../models/best.pt` for use in detection and tracking pipelines